# PERMUTATION TESTS

## These test the notion that a strategy is stronger within subject then between subject

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

from Markov_Chains.markov_data_manager import ParticipantMarkov

# Define panels
VALID_PANELS = ["0", "i1", "l4", "a3", "a5", "l3"]

def get_global_rois(participants_list):
    """
    1. Scans ALL participants.
    2. Returns a sorted list of unique ROIs (States).
    3. Forces all to strings to prevent 'int vs str' errors.
    """
    unique_rois = set()
    for p in participants_list:
        for panel, matrix in p.matrices.items():
            if matrix is not None:
                # Force conversion to string to handle mixed types (e.g. 1 vs "1")
                unique_rois.update(matrix.index.astype(str).tolist())
                unique_rois.update(matrix.columns.astype(str).tolist())
    
    # Filter out potential None/NaN string artifacts
    clean_rois = [x for x in unique_rois if x.lower() not in ['nan', 'none']]
    return sorted(clean_rois)

def align_and_flatten(df, global_rois):
    """
    Aligns a participant's matrix to the global ROI set and flattens it.
    """
    if df is None:
        return None
    
    # 1. Ensure the matrix also uses string indices/columns to match Global ROIs
    df_str = df.copy()
    df_str.index = df_str.index.astype(str)
    df_str.columns = df_str.columns.astype(str)
    
    # 2. Reindex to Global Set (fills missing spots with 0)
    df_aligned = df_str.reindex(index=global_rois, columns=global_rois, fill_value=0)
    
    return df_aligned.values.flatten()

def calc_mean_within_subject_correlation_real(participants_list: list[ParticipantMarkov]):
    """
    Helper function to calculate the mean within-subject correlation or extract them from csv
    """
    all_corrs = []
    
    for participant in participants_list:
        csv_path = os.path.join(participant.path, f'within_subject_correlation_{participant.group}_{participant.name}.csv')
        
        # Load DataFrame
        corr_matrix = pd.read_csv(csv_path, index_col=0)
        
        # FIX: Convert to NumPy array (.values) to allow integer indexing
        matrix_data = corr_matrix.values
        
        # Get indices for the upper triangle
        rows, cols = np.triu_indices_from(matrix_data, k=1)
        
        # Select values using the NumPy array, NOT the DataFrame
        upper_values = matrix_data[rows, cols]
        
        subject_mean = np.nanmean(upper_values)
        all_corrs.append(subject_mean)
        
    return np.nanmean(all_corrs), all_corrs

def calc_mean_within_subject_correlation_shuffeled(current_pools, n_subs, panel_list):
    all_corrs = []
    
    for i in range(n_subs):
        subject_matrices = []
        for panel in panel_list:
            m = current_pools[panel][i]
            if m is not None:
                subject_matrices.append(m)
        
        # Need at least 2 matrices to correlate
        if len(subject_matrices) < 2:
            continue
            
        stack = np.array(subject_matrices, dtype=float)
        
        # --- CRITICAL FIX: Remove Zero-Variance Vectors ---
        # If a matrix is all zeros (or constant), corrcoef returns NaN.
        # We keep only rows where standard deviation > 0
        valid_rows = stack.std(axis=1) > 1e-9
        stack = stack[valid_rows]
        
        if len(stack) < 2:
            continue

        # Calculate Correlation
        corr_mat = np.corrcoef(stack)
        
        # Extract off-diagonal upper triangle
        idx = np.triu_indices_from(corr_mat, k=1)
        valid_vals = corr_mat[idx]
        
        # If we have valid correlations, add the mean
        if len(valid_vals) > 0:
            subject_mean = np.nanmean(valid_vals)
            all_corrs.append(subject_mean)
        
    return np.nanmean(all_corrs)

def run_permutation_test(mdm_instance, n_permutations=1000):
    print(f"--- Starting Permutation Test ({n_permutations} iterations) ---")
    
    # 1. Collect participants
    all_participants = []
    for group in mdm_instance.participants.keys():
        for _, participant in mdm_instance.participants[group].items():
            all_participants.append(participant)
            
    n_subs = len(all_participants)
    
    # 2. Global Alignment
    global_rois = get_global_rois(all_participants)
    print(f"Global ROI Set ({len(global_rois)}): {global_rois}")
    
    # 3. Build Pools
    pools = {}
    for panel in VALID_PANELS:
        pools[panel] = []
        for p in all_participants:
            mat = p.matrices.get(panel)
            if mat is not None:
                flat = align_and_flatten(mat, global_rois)
                pools[panel].append(flat)
            else:
                pools[panel].append(None)

    # 4. Observed Statistic
    observed_stat = calc_mean_within_subject_correlation_real(all_participants)[0]
    print(f"Observed Mean Within-Subject Correlation: {observed_stat:.4f}")

    # 5. Permutation Loop
    null_distribution = []
    numpy_pools = {k: np.array(v, dtype=object) for k, v in pools.items()}

    for _ in tqdm(range(n_permutations)):
        shuffled_pools = {}
        for panel in VALID_PANELS:
            arr = numpy_pools[panel].copy()
            np.random.shuffle(arr) 
            shuffled_pools[panel] = arr
            
        perm_score = calc_mean_within_subject_correlation_shuffeled(shuffled_pools, n_subs, VALID_PANELS)
        null_distribution.append(perm_score)

    null_distribution = np.array(null_distribution)

    # 6. P-Value (ignoring NaNs in count)
    valid_nulls = null_distribution[~np.isnan(null_distribution)]
    if len(valid_nulls) == 0:
        print("WARNING: All permutations returned NaN. Check data quality.")
        p_value = 1.0
    else:
        n_better = np.sum(valid_nulls >= observed_stat)
        p_value = (n_better + 1) / (len(valid_nulls) + 1)
    
    print(f"Permutation P-Value: {p_value:.5f}")
    
    return observed_stat, null_distribution, p_value

def plot_permutation_results(observed, null_dist, p_val, output_path):
    plt.figure(figsize=(10, 6))
    
    # --- FIX: Filter NaNs before plotting ---
    clean_dist = null_dist[~np.isnan(null_dist)]
    
    if len(clean_dist) == 0:
        print("Cannot plot: Null distribution contains no valid numbers.")
        plt.close()
        return

    # Histogram
    plt.hist(clean_dist, bins=30, color='gray', alpha=0.7, label='Null Distribution')
    
    # Observed Line
    plt.axvline(observed, color='red', linestyle='--', linewidth=3, label=f'Observed (r={observed:.3f})')
    
    plt.title(f"Permutation Test Results\n(p-value: {p_val:.5f})")
    plt.xlabel("Mean Within-Subject Correlation")
    plt.ylabel("Frequency")
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.savefig(output_path)
    plt.close()
    print(f"Permutation plot saved to: {output_path}")

In [ ]:
from Markov_Chains.markov_data_manager import MarkovDataManager
import os
BEHAVIOR_ROOT = "/Volumes/ramot/Noam_M/Results/Behavior"
MATRIX_ROOT = "/Volumes/ramot/Noam_M/preliminary_results"

WITH_REPEATS = False 


def main():
    # ... (Load data as before) ...
    mdm = MarkovDataManager(BEHAVIOR_ROOT, MATRIX_ROOT, with_repeats=WITH_REPEATS)
    mdm.load_all_data(from_scratch=False)

    # ... (Run standard analysis) ...

    # --- NEW: Run Permutation Test ---
    print("\n--- Running Permutation Test ---")
    obs, null_dist, p_val = run_permutation_test(mdm, n_permutations=1000)
    
    plot_path = os.path.join(MATRIX_ROOT, "no repeats", "permutation_test_result.png")
    plot_permutation_results(obs, null_dist, p_val, plot_path)

    if p_val < 0.05:
        print("RESULT: Significant! Participants have distinct, consistent gaze strategies.")
    else:
        print("RESULT: Not Significant. Individual consistency is not better than random.")

In [ ]:
main()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

from Markov_Chains.markov_data_manager import ParticipantMarkov

VALID_PANELS = ["0", "i1", "l4", "a3", "a5", "l3"]

# --- 1. Same Helpers as before ---
def get_global_rois(participants_list):
    unique_rois = set()
    for p in participants_list:
        for panel, matrix in p.matrices.items():
            if matrix is not None:
                unique_rois.update(matrix.index.astype(str).tolist())
                unique_rois.update(matrix.columns.astype(str).tolist())
    clean_rois = [x for x in unique_rois if x.lower() not in ['nan', 'none']]
    return sorted(clean_rois)

def align_and_flatten(df, global_rois):
    if df is None: return None
    df_str = df.copy()
    df_str.index = df_str.index.astype(str)
    df_str.columns = df_str.columns.astype(str)
    df_aligned = df_str.reindex(index=global_rois, columns=global_rois, fill_value=0)
    return df_aligned.values.flatten()

def calc_mean_within_subject_correlation_real(participants_list: list[ParticipantMarkov]):
    """
    Helper function to calculate the mean within-subject correlation or extract them from csv
    """
    all_corrs = []
    
    for participant in participants_list:
        csv_path = os.path.join(participant.path, f'within_subject_correlation_{participant.group}_{participant.name}.csv')
        
        # Load DataFrame
        corr_matrix = pd.read_csv(csv_path, index_col=0)
        
        # FIX: Convert to NumPy array (.values) to allow integer indexing
        matrix_data = corr_matrix.values
        
        # Get indices for the upper triangle
        rows, cols = np.triu_indices_from(matrix_data, k=1)
        
        # Select values using the NumPy array, NOT the DataFrame
        upper_values = matrix_data[rows, cols]
        
        subject_mean = np.nanmean(upper_values)
        all_corrs.append(subject_mean)
        
    return np.nanmean(all_corrs), all_corrs

# --- 2. UPDATED Calculation Function ---
def calc_stats_and_values(current_pools, n_subs, panel_list):
    """
    Returns a tuple: (Mean_Correlation, List_of_Individual_Correlations)
    """
    individual_corrs = []
    
    for i in range(n_subs):
        subject_matrices = []
        for panel in panel_list:
            m = current_pools[panel][i]
            if m is not None:
                subject_matrices.append(m)
        
        if len(subject_matrices) < 2:
            continue
            
        stack = np.array(subject_matrices, dtype=float)
        
        # Remove zero-variance rows
        valid_rows = stack.std(axis=1) > 1e-9
        stack = stack[valid_rows]
        
        if len(stack) < 2:
            continue

        corr_mat = np.corrcoef(stack)
        idx = np.triu_indices_from(corr_mat, k=1)
        valid_vals = corr_mat[idx]
        
        if len(valid_vals) > 0:
            subject_mean = np.nanmean(valid_vals)
            individual_corrs.append(subject_mean)
    
    if len(individual_corrs) == 0:
        return np.nan, []
        
    return np.nanmean(individual_corrs), individual_corrs

# --- 3. UPDATED Permutation Runner ---
def run_permutation_test(mdm_instance, n_permutations=1000):
    print(f"--- Starting Permutation Test ({n_permutations} iterations) ---")
    
    # Setup Data
    all_participants = []
    for group in mdm_instance.participants.keys():
        for _, participant in mdm_instance.participants[group].items():
            all_participants.append(participant)
            
    n_subs = len(all_participants)
    global_rois = get_global_rois(all_participants)
    
    # Build Aligned Pools
    pools = {}
    for panel in VALID_PANELS:
        pools[panel] = []
        for p in all_participants:
            mat = p.matrices.get(panel)
            if mat is not None:
                flat = align_and_flatten(mat, global_rois)
                pools[panel].append(flat)
            else:
                pools[panel].append(None)

    # --- Real Data Calculation ---
    # We now capture both the Mean and the individual values
    obs_mean, obs_values = calc_mean_within_subject_correlation_real(all_participants)
    print(f"Observed Mean: {obs_mean:.4f}")

    # --- Permutation Loop ---
    null_means = []           # For P-value calculation (Distribution of Means)
    all_random_values = []    # For Plotting (Distribution of Individuals)
    
    numpy_pools = {k: np.array(v, dtype=object) for k, v in pools.items()}

    for _ in tqdm(range(n_permutations)):
        shuffled_pools = {}
        for panel in VALID_PANELS:
            arr = numpy_pools[panel].copy()
            np.random.shuffle(arr) 
            shuffled_pools[panel] = arr
            
        perm_mean, perm_vals = calc_stats_and_values(shuffled_pools, n_subs, VALID_PANELS)
        
        null_means.append(perm_mean)
        all_random_values.extend(perm_vals) # Collect all random scores for the histogram

    null_means = np.array(null_means)

    # P-Value (Using the Means)
    valid_nulls = null_means[~np.isnan(null_means)]
    n_better = np.sum(valid_nulls >= obs_mean)
    p_value = (n_better + 1) / (len(valid_nulls) + 1)
    
    print(f"Permutation P-Value: {p_value:.5f}")
    
    # Return everything needed for the new plot
    return obs_mean, obs_values, null_means, all_random_values, p_value

# --- 4. UPDATED Plotter ---
def plot_permutation_results(obs_mean, obs_values, random_values, p_val, output_path):
    plt.figure(figsize=(10, 6))
    
    # Clean NaNs
    clean_obs = [x for x in obs_values if not np.isnan(x)]
    clean_rnd = [x for x in random_values if not np.isnan(x)]
    
    # Calculate means for the legend
    mean_obs = np.mean(clean_obs)
    mean_rnd = np.mean(clean_rnd)
    
    # Plot 1: Random Distribution (Gray)
    # We use density=True because we have WAY more random points (N_subs * N_perms) 
    # than observed points (N_subs). Density allows us to compare the shapes.
    plt.hist(clean_rnd, bins=50, color='gray', alpha=0.5, density=True, 
             label=f'Random Shuffles (Mean={mean_rnd:.3f})')
    
    # Plot 2: Observed Distribution (Red)
    plt.hist(clean_obs, bins=15, color='red', alpha=0.5, density=True, 
             label=f'Real Participants (Mean={mean_obs:.3f})')
    
    # Vertical Lines for Means
    plt.axvline(mean_rnd, color='gray', linestyle='--', linewidth=2)
    plt.axvline(mean_obs, color='red', linestyle='--', linewidth=2)
    
    plt.title(f"Within-Subject Consistency: Real vs Random\n(Permutation p-value: {p_val:.5f})")
    plt.xlabel("Correlation Coefficient (r)")
    plt.ylabel("Density")
    plt.legend(loc='upper left')
    plt.grid(True, alpha=0.3)
    
    plt.savefig(output_path)
    plt.close()
    print(f"Enhanced plot saved to: {output_path}")

# --- 5. Main Execution ---
def main():
    # ... (Load data as usual) ...
    from Markov_Chains.markov_data_manager import MarkovDataManager
    # Define paths
    BEHAVIOR_ROOT = "/Volumes/ramot/Noam_M/Results/Behavior"
    MATRIX_ROOT = "/Volumes/ramot/Noam_M/preliminary_results"
    
    mdm = MarkovDataManager(BEHAVIOR_ROOT, MATRIX_ROOT, with_repeats=False)
    mdm.load_all_data(from_scratch=False)

    # Run Test
    obs_mean, obs_vals, null_means, rnd_vals, p_val = run_permutation_test(mdm, n_permutations=1000)
    
    # Plot
    plot_path = os.path.join(MATRIX_ROOT, "permutation_distribution.png")
    plot_permutation_results(obs_mean, obs_vals, rnd_vals, p_val, plot_path)

if __name__ == "__main__":
    main()

Loading participants...
Loading scores...
Loading Markov matrices...
[WARNING] No matrices loaded for HC/LE750, removing participant.
[WARNING] No matrices loaded for HC/LT157, removing participant.
[WARNING] No matrices loaded for HC/EE050, removing participant.
[WARNING] No matrices loaded for HC/FZ767, removing participant.
[WARNING] No matrices loaded for HC/KT158, removing participant.
[WARNING] No matrices loaded for HC/ZM425, removing participant.
[WARNING] No matrices loaded for HC/RL823, removing participant.
[WARNING] No matrices loaded for HC/YR187, removing participant.
[WARNING] No matrices loaded for HC/DM242, removing participant.
[WARNING] No matrices loaded for HC/KS689, removing participant.
[WARNING] No matrices loaded for HC/LY082, removing participant.
[WARNING] No matrices loaded for HC/KS130, removing participant.
[WARNING] No matrices loaded for HC/PR454, removing participant.
[WARNING] No matrices loaded for HC/BO921, removing participant.
[WARNING] No matrices

100%|██████████| 1000/1000 [00:06<00:00, 143.36it/s]


Permutation P-Value: 0.00100
Enhanced plot saved to: /Volumes/ramot/Noam_M/preliminary_results/permutation_distribution.png


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import os

# --- Import your existing classes ---
# Assuming the code you pasted is in a file or available in the current namespace.
# If they are in 'markov_data_manager.py', uncomment the line below:
from Markov_Chains.markov_data_manager import MarkovDataManager, ParticipantMarkov

class MeanMatrixPCA:
    """
    Analyzes the Mean Markov Matrix of each participant using PCA,
    coloring the results by their average SDMT score.
    """
    def __init__(self, mdm_instance):
        self.mdm = mdm_instance
        self.feature_matrix = None  # (N_subjects, N_features)
        self.scores = None          # (N_subjects, )
        self.names = None           # (N_subjects, )
        self.groups = None          # (N_subjects, )
        self.pca_coords = None

    def _get_global_rois(self, participants):
        """
        Finds the union of all ROIs across all participants' mean matrices
        to ensure feature vectors are aligned.
        """
        unique_rois = set()
        for p in participants:
            if p.mean_matrix is not None:
                # Force to string to avoid int vs str type errors
                unique_rois.update(p.mean_matrix.index.astype(str).tolist())
                unique_rois.update(p.mean_matrix.columns.astype(str).tolist())
        
        # Filter NaNs/None
        clean = [r for r in unique_rois if r.lower() not in ['nan', 'none']]
        return sorted(clean)

    def _align_and_flatten(self, df, global_rois):
        """Aligns a matrix to the global ROI set and flattens it."""
        if df is None: return None
        
        # Ensure string types for matching
        df_str = df.copy()
        df_str.index = df_str.index.astype(str)
        df_str.columns = df_str.columns.astype(str)
        
        # Reindex to force consistent shape (N_rois x N_rois)
        # Fill missing states with 0 probability
        df_aligned = df_str.reindex(index=global_rois, columns=global_rois, fill_value=0)
        
        return df_aligned.values.flatten()

    def prepare_data(self):
        print("Preparing data for Mean Matrix PCA...")
        
        # 1. Collect all valid participants
        all_participants = []
        for group in self.mdm.participants:
            for p_name, p_obj in self.mdm.participants[group].items():
                if p_obj.mean_matrix is not None:
                    all_participants.append(p_obj)
        
        if not all_participants:
            print("No participants with mean matrices found.")
            return

        # 2. Determine Global Alignment (crucial for PCA)
        global_rois = self._get_global_rois(all_participants)
        print(f"Global ROI Set ({len(global_rois)}): {global_rois}")

        vectors = []
        scores_list = []
        names_list = []
        groups_list = []

        # 3. Extract Data
        for p in all_participants:
            # A. Vectorize Matrix
            vec = self._align_and_flatten(p.mean_matrix, global_rois)
            
            # B. Calculate Mean Score
            # p.scores is {panel: score}. We take the average.
            if p.scores:
                # Filter out None values just in case
                valid_scores = [s for s in p.scores.values() if s is not None]
                avg_score = np.mean(valid_scores) if valid_scores else np.nan
            else:
                avg_score = np.nan

            vectors.append(vec)
            scores_list.append(avg_score)
            names_list.append(p.name)
            groups_list.append(p.group)

        self.feature_matrix = np.array(vectors)
        self.scores = np.array(scores_list)
        self.names = np.array(names_list)
        self.groups = np.array(groups_list)
        
        print(f"Data prepared. Shape: {self.feature_matrix.shape}")

    def run_pca_and_plot(self, output_path):
        if self.feature_matrix is None:
            self.prepare_data()

        # Filter out participants with NaN scores (cannot color them)
        # Alternatively, we could plot them in grey, but let's filter for clarity
        valid_mask = ~np.isnan(self.scores)
        
        X = self.feature_matrix[valid_mask]
        y = self.scores[valid_mask]
        current_names = self.names[valid_mask]
        
        if len(X) < 3:
            print("Not enough data points with valid scores for PCA.")
            return

        # 1. Standardize (Optional for probability matrices, but usually good for PCA)
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        # 2. PCA
        pca = PCA(n_components=2)
        coords = pca.fit_transform(X_scaled)
        
        explained_variance = pca.explained_variance_ratio_

        # 3. Plotting
        plt.figure(figsize=(10, 8))
        
        # Scatter with colormap
        sc = plt.scatter(coords[:, 0], coords[:, 1], c=y, cmap='viridis', s=100, edgecolors='k', alpha=0.8)
        
        # Add colorbar
        cbar = plt.colorbar(sc)
        cbar.set_label('Mean SDMT Score', fontsize=12)

        # Annotate points (Optional - can get crowded)
        # for i, txt in enumerate(current_names):
        #     plt.annotate(txt, (coords[i, 0], coords[i, 1]), fontsize=8, alpha=0.7)

        plt.xlabel(f"PC1 ({explained_variance[0]:.1%} variance)", fontsize=12)
        plt.ylabel(f"PC2 ({explained_variance[1]:.1%} variance)", fontsize=12)
        plt.title("PCA of Mean Markov Matrices\nColored by Mean SDMT Score", fontsize=15)
        plt.grid(True, alpha=0.3)
        
        # Save
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        plt.savefig(output_path)
        plt.close()
        print(f"PCA Plot saved to: {output_path}")

# ============================================================
# RUNNER
# ============================================================

def main_pca_workflow():
    # 1. Config
    BEHAVIOR_ROOT = "/Volumes/ramot/Noam_M/Results/Behavior"
    MATRIX_ROOT = "/Volumes/ramot/Noam_M/preliminary_results"
    WITH_REPEATS = False 

    # 2. Load Data using your existing Manager
    print("--- Initializing Data Manager ---")
    mdm = MarkovDataManager(BEHAVIOR_ROOT, MATRIX_ROOT, with_repeats=WITH_REPEATS)
    mdm.load_all_data(from_scratch=False)

    # 3. Run the New PCA Analysis
    print("\n--- Running Mean Matrix PCA ---")
    pca_analyzer = MeanMatrixPCA(mdm)
    
    output_file = os.path.join(MATRIX_ROOT, "pca_mean_matrix_by_score.png")
    pca_analyzer.run_pca_and_plot(output_file)

if __name__ == "__main__":
    main_pca_workflow()

--- Initializing Data Manager ---
Loading participants...
Loading scores...
Loading Markov matrices...
[WARNING] No matrices loaded for HC/LE750, removing participant.
[WARNING] No matrices loaded for HC/LT157, removing participant.
[WARNING] No matrices loaded for HC/EE050, removing participant.
[WARNING] No matrices loaded for HC/FZ767, removing participant.
[WARNING] No matrices loaded for HC/KT158, removing participant.
[WARNING] No matrices loaded for HC/ZM425, removing participant.
[WARNING] No matrices loaded for HC/RL823, removing participant.
[WARNING] No matrices loaded for HC/YR187, removing participant.
[WARNING] No matrices loaded for HC/DM242, removing participant.
[WARNING] No matrices loaded for HC/KS689, removing participant.
[WARNING] No matrices loaded for HC/LY082, removing participant.
[WARNING] No matrices loaded for HC/KS130, removing participant.
[WARNING] No matrices loaded for HC/PR454, removing participant.
[WARNING] No matrices loaded for HC/BO921, removing 